# 01 — Data Exploration

**Goal:** Load the Kaggle dataset, understand structure, identify unique entities.

**Checkpoint:** Know exact counts of plant_codes, asset_tags, part_nos and their relationships.

**Sections:**
1. Load raw data — `.info()` and `.describe()`
2. Sensor feature distributions (histograms)
3. Breakdown rate analysis
4. Entity structure
5. Correlation heatmap
6. Time series of breakdowns

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

np.random.seed(42)

# Walk up from CWD until we find the project root (contains src/ and data/).
# This is robust whether the notebook is launched from Prototype/ or notebooks/.
_cwd = Path(os.getcwd())
_candidate = _cwd
ROOT = None
for _ in range(4):
    if (_candidate / "src").is_dir() and (_candidate / "data").is_dir():
        ROOT = _candidate
        break
    _candidate = _candidate.parent
if ROOT is None:
    raise RuntimeError(f"[setup] Could not find project root from {_cwd}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.loader import load_csv

# ── diagnostics ────────────────────────────────────────────────────────────────────────
_raw_dir = ROOT / "data" / "raw"
print(f"[setup] project root   : {ROOT}")
print(f"[setup] CWD            : {_cwd}")
print(f"[setup] data/raw exists: {_raw_dir.exists()}")
if _raw_dir.exists():
    _files = sorted(p.name for p in _raw_dir.iterdir())
    print(f"[setup] files in data/raw: {_files}")
else:
    print(f"[setup] WARNING — data/raw not found at {_raw_dir}")

# ── constants ──────────────────────────────────────────────────────────────
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BG_COLOR   = "#0a0e17"
TEXT_COLOR = "white"
GRID_COLOR = "#1e2636"

SENSOR_COLS = [
    "temp_bearing_degC",
    "temp_motor_degC",
    "vibration_h_mms",
    "vibration_v_mms",
    "oil_pressure_bar",
    "load_pct",
    "power_consumption_kw",
    "shaft_rpm",
]

# Colour palette per layer (CODING_PATTERNS.md)
LAYER_COLORS = {
    "supplier":     "#3a86ff",
    "logistics":    "#9b5de5",
    "plant":        "#06d6a0",
    "machine":      "#ffbe0b",
    "distribution": "#ef233c",
}
SENSOR_COLOR  = LAYER_COLORS["machine"]   # amber — sensor data is machine-layer
BREAKDOWN_COLOR = "#ef233c"               # red — disruption / breakdown


def apply_dark_theme(fig: plt.Figure, axes) -> None:
    """Apply the project dark theme to a figure and all its axes."""
    fig.patch.set_facecolor(BG_COLOR)
    ax_list = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for ax in ax_list:
        ax.set_facecolor(BG_COLOR)
        ax.tick_params(colors=TEXT_COLOR)
        ax.xaxis.label.set_color(TEXT_COLOR)
        ax.yaxis.label.set_color(TEXT_COLOR)
        ax.title.set_color(TEXT_COLOR)
        for spine in ax.spines.values():
            spine.set_edgecolor(GRID_COLOR)
        ax.grid(True, color=GRID_COLOR, linewidth=0.5)


print("Setup complete.")

---
## 1. Load Raw Data

In [ ]:
# ── adjust filename to match your downloaded CSV ───────────────────────────
CSV_FILENAME = "updated_data.csv"

df_raw: pd.DataFrame = load_csv(CSV_FILENAME)
print(f"Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="all").T

---
## 2. Sensor Feature Distributions

In [ ]:
# Use only the first 6 sensor columns for the subplot grid (3×2)
SENSOR_PLOT_COLS = SENSOR_COLS[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
apply_dark_theme(fig, axes)
fig.suptitle("Sensor Feature Distributions (raw)", color=TEXT_COLOR, fontsize=14, y=1.01)

for ax, col in zip(axes.flatten(), SENSOR_PLOT_COLS):
    data = df_raw[col].dropna()
    ax.hist(data, bins=50, color=SENSOR_COLOR, edgecolor=BG_COLOR, linewidth=0.3)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")

plt.tight_layout()
out_path = RESULTS_DIR / "dtnet_sensor_distributions.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
print(f"Saved → {out_path}")
plt.show()

---
## 3. Breakdown Rate Analysis

In [ ]:
# ── overall breakdown rate ─────────────────────────────────────────────────
df_dated = df_raw.copy()
df_dated["transaction_date"] = pd.to_datetime(df_dated["transaction_date"], errors="coerce")
df_dated = df_dated.dropna(subset=["transaction_date"])
df_dated["date"] = df_dated["transaction_date"].dt.date

daily = df_dated.groupby("date")["breakdown_flag"].max().reset_index()
pct_breakdown_days = daily["breakdown_flag"].mean() * 100
print(f"Days with at least one breakdown : {daily['breakdown_flag'].sum():,}")
print(f"Total unique days                : {len(daily):,}")
print(f"Breakdown rate (day-level)       : {pct_breakdown_days:.2f}%")

In [ ]:
# ── breakdown rate per plant_code ──────────────────────────────────────────
plant_bd = (
    df_dated.groupby("plant_code")["breakdown_flag"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)
plant_bd.columns = ["plant_code", "breakdown_rate_pct"]
print(plant_bd.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
apply_dark_theme(fig, ax)
ax.bar(plant_bd["plant_code"].astype(str), plant_bd["breakdown_rate_pct"],
       color=BREAKDOWN_COLOR, edgecolor=BG_COLOR)
ax.set_title("Breakdown Rate (%) per Plant Code", fontsize=13)
ax.set_xlabel("plant_code")
ax.set_ylabel("Breakdown rate (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
plt.tight_layout()
out_path = RESULTS_DIR / "dtnet_breakdown_rate_by_plant.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
print(f"Saved → {out_path}")
plt.show()

In [ ]:
# ── breakdown rate per machine_type ───────────────────────────────────────
mtype_bd = (
    df_dated.groupby("machine_type")["breakdown_flag"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)
mtype_bd.columns = ["machine_type", "breakdown_rate_pct"]
print(mtype_bd.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
apply_dark_theme(fig, ax)
ax.bar(mtype_bd["machine_type"].astype(str), mtype_bd["breakdown_rate_pct"],
       color=LAYER_COLORS["machine"], edgecolor=BG_COLOR)
ax.set_title("Breakdown Rate (%) per Machine Type", fontsize=13)
ax.set_xlabel("machine_type")
ax.set_ylabel("Breakdown rate (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
out_path = RESULTS_DIR / "dtnet_breakdown_rate_by_machine_type.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
print(f"Saved → {out_path}")
plt.show()

---
## 4. Entity Structure

In [ ]:
# ── top-level unique counts ────────────────────────────────────────────────
for col in ["plant_code", "asset_tag", "part_no"]:
    if col in df_raw.columns:
        print(f"Unique {col:<15}: {df_raw[col].nunique():>6,}")
    else:
        print(f"Column '{col}' not found in dataset.")

In [ ]:
# ── machines per plant ─────────────────────────────────────────────────────
if "plant_code" in df_raw.columns and "asset_tag" in df_raw.columns:
    machines_per_plant = (
        df_raw.groupby("plant_code")["asset_tag"]
        .nunique()
        .sort_values(ascending=False)
        .reset_index()
    )
    machines_per_plant.columns = ["plant_code", "num_machines"]
    print(machines_per_plant.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 4))
    apply_dark_theme(fig, ax)
    ax.bar(machines_per_plant["plant_code"].astype(str),
           machines_per_plant["num_machines"],
           color=LAYER_COLORS["plant"], edgecolor=BG_COLOR)
    ax.set_title("Number of Unique Machines per Plant", fontsize=13)
    ax.set_xlabel("plant_code")
    ax.set_ylabel("Number of machines")
    plt.tight_layout()
    out_path = RESULTS_DIR / "dtnet_machines_per_plant.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
    print(f"Saved → {out_path}")
    plt.show()

In [ ]:
# ── parts per machine ──────────────────────────────────────────────────────
if "asset_tag" in df_raw.columns and "part_no" in df_raw.columns:
    parts_per_machine = (
        df_raw.groupby("asset_tag")["part_no"]
        .nunique()
        .reset_index()
    )
    parts_per_machine.columns = ["asset_tag", "num_parts"]

    print(f"Parts per machine — stats:")
    print(parts_per_machine["num_parts"].describe().to_string())

    fig, ax = plt.subplots(figsize=(10, 4))
    apply_dark_theme(fig, ax)
    ax.hist(parts_per_machine["num_parts"], bins=30,
            color=LAYER_COLORS["logistics"], edgecolor=BG_COLOR, linewidth=0.3)
    ax.set_title("Distribution of Parts per Machine", fontsize=13)
    ax.set_xlabel("Number of unique parts")
    ax.set_ylabel("Number of machines")
    plt.tight_layout()
    out_path = RESULTS_DIR / "dtnet_parts_per_machine.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
    print(f"Saved → {out_path}")
    plt.show()

---
## 5. Correlation Heatmap — Sensors vs breakdown_flag

In [ ]:
available_sensors = [c for c in SENSOR_COLS if c in df_raw.columns]
corr_cols = available_sensors + ["breakdown_flag"]
corr_matrix = df_raw[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
apply_dark_theme(fig, ax)

cmap = matplotlib.cm.get_cmap("RdYlGn")   # diverging: red=negative, green=positive
im = ax.imshow(corr_matrix.values, cmap=cmap, vmin=-1, vmax=1, aspect="auto")

# Axis labels
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=9, color=TEXT_COLOR)
ax.set_yticklabels(corr_cols, fontsize=9, color=TEXT_COLOR)

# Annotate cells
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        val = corr_matrix.values[i, j]
        text_col = "black" if abs(val) > 0.5 else TEXT_COLOR
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=8, color=text_col)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COLOR)

ax.set_title("Pearson Correlation — Sensor Features & Breakdown Flag", fontsize=13)
plt.tight_layout()
out_path = RESULTS_DIR / "dtnet_sensor_correlation_heatmap.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
print(f"Saved → {out_path}")
plt.show()

---
## 6. Time Series of Breakdowns (3-year period)

In [ ]:
# ── monthly breakdown count ────────────────────────────────────────────────
df_ts = df_dated.copy()
df_ts["year_month"] = df_ts["transaction_date"].dt.to_period("M")

monthly_bd = (
    df_ts.groupby("year_month")["breakdown_flag"]
    .sum()
    .reset_index()
)
monthly_bd["year_month_dt"] = monthly_bd["year_month"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
apply_dark_theme(fig, ax)

ax.fill_between(monthly_bd["year_month_dt"], monthly_bd["breakdown_flag"],
                color=BREAKDOWN_COLOR, alpha=0.7, linewidth=0)
ax.plot(monthly_bd["year_month_dt"], monthly_bd["breakdown_flag"],
        color=BREAKDOWN_COLOR, linewidth=1.2)

ax.set_title("Monthly Breakdown Events Over the 3-Year Period", fontsize=13)
ax.set_xlabel("Month")
ax.set_ylabel("Number of breakdown events")
ax.xaxis.set_major_locator(matplotlib.dates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%b %Y"))
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
out_path = RESULTS_DIR / "dtnet_breakdown_timeseries.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
print(f"Saved → {out_path}")
plt.show()

In [ ]:
# ── weekly breakdown count (finer granularity) ─────────────────────────────
df_ts["year_week"] = df_ts["transaction_date"].dt.to_period("W")

weekly_bd = (
    df_ts.groupby("year_week")["breakdown_flag"]
    .sum()
    .reset_index()
)
weekly_bd["year_week_dt"] = weekly_bd["year_week"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
apply_dark_theme(fig, ax)

ax.fill_between(weekly_bd["year_week_dt"], weekly_bd["breakdown_flag"],
                color=LAYER_COLORS["machine"], alpha=0.6, linewidth=0)
ax.plot(weekly_bd["year_week_dt"], weekly_bd["breakdown_flag"],
        color=LAYER_COLORS["machine"], linewidth=1.0)

ax.set_title("Weekly Breakdown Events Over the 3-Year Period", fontsize=13)
ax.set_xlabel("Week")
ax.set_ylabel("Number of breakdown events")
ax.xaxis.set_major_locator(matplotlib.dates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%b %Y"))
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
out_path = RESULTS_DIR / "dtnet_breakdown_timeseries_weekly.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
print(f"Saved → {out_path}")
plt.show()

---
## Summary

| Item | Value |
|------|-------|
| Total rows | _(run cells above)_ |
| Unique plant_codes | _(run cells above)_ |
| Unique asset_tags | _(run cells above)_ |
| Unique part_nos | _(run cells above)_ |
| Overall breakdown rate | _(run cells above)_ |

**Next:** `02_graph_construction.ipynb` — build the DTNet supply-chain graph from these entities.